# Market Exploration — 무선이어폰 시장

This is where I prototype things before wiring them into the agent.
I scraped Naver Shopping manually here first to understand the data structure,
then moved the working code into `platforms/naver.py`.

The Codex scraper_gen stuff is tested at the bottom — it worked but I kept
hitting rate limits so I couldn't run many experiments.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv

load_dotenv()
print('Setup done')

## 1. Quick Naver Shopping scrape test

First I just wanted to see if I could get any data at all without Selenium.
Turns out Naver returns decent HTML with basic requests headers.

In [ ]:
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36',
    'Accept-Language': 'ko-KR,ko;q=0.9',
    'Referer': 'https://shopping.naver.com/',
}

resp = requests.get(
    'https://search.shopping.naver.com/search/all',
    params={'query': '무선이어폰', 'sort': 'review', 'pagingIndex': 1},
    headers=HEADERS,
    timeout=10
)
print('Status:', resp.status_code)
print('Content length:', len(resp.text))

In [ ]:
soup = BeautifulSoup(resp.text, 'html.parser')

# The class names change sometimes - this was correct as of May 2025
# If this breaks, inspect element on the page and find the new class
items = soup.select('li.product_item__MDtDF')
print(f'Found {len(items)} product items')

In [ ]:
# Parse first item to see what we're working with
item = items[0]

title   = item.select_one('.product_title__Mmn7R a')
price   = item.select_one('.price_num__S2p_v')
reviews = item.select_one('.product_grade__Frprs em')

print('Title:',   title.get_text(strip=True) if title else 'N/A')
print('Price:',   price.get_text(strip=True) if price else 'N/A')
print('Reviews:', reviews.get_text(strip=True) if reviews else 'N/A')

## 2. Collect a small dataset

Running 2 pages to get ~80 products. Moved this into `platforms/naver.search()` afterwards.

In [ ]:
import sys
sys.path.append('..')

from platforms.naver import search

products = search('무선이어폰', sort='review', max_pages=2)
print(f'Got {len(products)} products')

df = pd.DataFrame([
    {
        'title':        p.title,
        'price':        p.price,
        'review_count': p.review_count,
        'rating':       p.rating,
        'seller':       p.seller,
        'is_ad':        p.is_ad,
    }
    for p in products
])
df.head()

## 3. Basic price distribution

In [ ]:
prices = df['price'].dropna()

print('Mean price:   ', int(prices.mean()), 'KRW')
print('Median price: ', int(prices.median()), 'KRW')
print('Min:          ', int(prices.min()), 'KRW')
print('Max:          ', int(prices.max()), 'KRW')
print()
print('Price quantiles:')
print(prices.quantile([0.25, 0.5, 0.75]).apply(lambda x: f'{int(x):,} KRW'))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(prices / 1000, bins=20, color='#5C7CFA', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Price (천원, KRW thousands)')
ax.set_ylabel('Product count')
ax.set_title('Naver Shopping — Wireless Earphone Price Distribution')
plt.tight_layout()
plt.savefig('price_distribution.png', dpi=150)
plt.show()

## 4. Codex scraper generation test

This is where I tested whether Codex can write a scraper from a description.
It worked — the generated code wasn't perfect but it ran and returned data.

**Problem:** code-davinci-002 is expensive. Each call for a 600-token completion
costs a meaningful chunk of credits. I was testing with maybe 20-30 iterations
and ran out. Commented out so it doesn't accidentally run and burn tokens.

In [ ]:
# from codex.scraper_gen import generate_scraper, run_generated_code
# import json
#
# code = generate_scraper(
#     platform='11번가',
#     keyword='무선이어폰',
#     fields=['title', 'price', 'review_count', 'link'],
#     max_results=20
# )
#
# print('Generated code preview:')
# print(code[:500])
# print('---')
#
# output = run_generated_code(code)
# products = json.loads(output)
# print(f'Got {len(products)} products from generated scraper')

print('Codex test commented out - needs API balance to run')
print('Last successful run: 2025-05-12, got 18 products from 11번가')

## 5. What I want to do next

- Run this same search on Coupang and compare prices side by side
- Track prices over time (run every few days, store in SQLite)
- Use Codex to generate the comparison analysis automatically instead of writing it manually
- Test with other product categories I'm considering

The limiting factor right now is API credits for the Codex calls.
The scraping parts (Naver, Coupang) work fine locally.